# ◈ Parlay — SFT Training on Qwen2.5-1.5B
## Teaching a language model to negotiate

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** on Parlay negotiation
episodes using **Supervised Fine-Tuning (SFT)** with LoRA via
[Unsloth](https://github.com/unslothai/unsloth).

**What you need**
- A free Google Colab **T4** GPU
- Colab secret `HF_TOKEN` (write access)

**What this does**
1. Installs stack + checks GPU
2. Loads `sh4shv4t/parlay-episodes` from the Hub
3. Formats to ChatML, trains with LoRA (r=16)
4. Plots, evaluates before/after, pushes adapter to `sh4shv4t/parlay-negotiator`

**Runtime:** ~25 min on T4 · **Cost:** $0 (free tier).

**If `load_dataset` fails on `Feature type 'Json'`:** the install cell upgrades `datasets`;
if it still fails, use **Runtime → Restart runtime**, then run from Step 2 onward. The
data cell also falls back to downloading `episodes_v2.jsonl` directly (no `Json` schema).


## Step 1 — Install dependencies


In [1]:
%%capture
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "--quiet",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "trl>=0.8.6", "peft>=0.10.0", "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0", "xformers", "--quiet",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-U", "datasets>=3.0.0", "huggingface-hub>=0.23.0", "pyarrow>=14.0.0",
    "--quiet",
], check=True)
print("OK: dependencies (datasets>=3 for Json dtype on Hub)")


## Step 2 — GPU + config


In [2]:
import os, json
import torch
from google.colab import userdata

# Before any `import unsloth` in later cells — avoids TimeoutError on HF stats ping
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible to PyTorch. Colab: Runtime → Change runtime type → set "
        "Hardware accelerator to GPU (T4, L4, A100, …) → Save, then re-run. "
        "If you are on **CPU** only, this notebook cannot train; connect a GPU session."
    )
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1e9, "GB VRAM")
print("torch", torch.__version__, "cuda", torch.version.cuda)

HF_TOKEN = userdata.get("HF_TOKEN")
DATASET_ID = "sh4shv4t/parlay-episodes"
MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct"
OUTPUT_REPO = "sh4shv4t/parlay-negotiator"
LORA_R, LORA_ALPHA = 16, 32
MAX_SEQ_LEN = 1024
SFT_EPOCHS = 2
BATCH_SIZE, GRAD_ACCUM = 2, 4
LEARNING_RATE, WARMUP_RATIO = 2e-4, 0.1
MIN_REWARD_KEEP = 0.25


Tesla T4 15.637086208 GB VRAM
torch 2.10.0+cu128 cuda 12.8


## Step 3 — Load dataset


In [3]:
import json as _json
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import hf_hub_download


def load_parlay_episodes():
    """Prefer `load_dataset`; fall back to raw JSONL if Colab's old `datasets` can't parse `Json` features."""
    try:
        return load_dataset(DATASET_ID, token=HF_TOKEN)
    except (ValueError, KeyError) as e:
        msg = str(e)
        if "Json" not in msg and "Feature type" not in msg:
            raise
        path = hf_hub_download(
            repo_id=DATASET_ID,
            repo_type="dataset",
            filename="episodes_v2.jsonl",
            token=HF_TOKEN,
        )
        rows = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(_json.loads(line))
        df = pd.DataFrame(rows)
        return DatasetDict({"train": Dataset.from_pandas(df, preserve_index=False)})


raw = load_parlay_episodes()
df = raw["train"].to_pandas()
print(len(df), "rows", list(df.columns))
if "reward" in df.columns:
    print(df["reward"].describe())


140 rows ['prompt', 'conversation', 'reward', 'deal_efficiency', 'persona', 'scenario_id', 'acts_completed', 'tom_accuracy', 'tom_accuracy_avg', 'drift_adapted', 'split', 'deal_reached', 'episode_id', 'termination_reason', 'batna_seller', 'batna_buyer']
count    140.000000
mean      65.009969
std       42.698799
min      -49.317772
25%       43.282352
50%       71.315162
75%       95.505128
max      131.133467
Name: reward, dtype: float64


## Step 4 — Format episodes (ChatML)


In [4]:
from datasets import Dataset

SYSTEM = """You negotiate in a B2B deal. Respond with JSON only:
{"utterance": str, "offer_amount": number|null, "tactical_move": str|null}"""

def format_episode(row):
    conv = row.get("conversation", row.get("messages", []))
    if isinstance(conv, str):
        try:
            conv = json.loads(conv)
        except Exception:
            return None
    if not isinstance(conv, list) or len(conv) < 2:
        return None
    msgs = [{"role": "system", "content": SYSTEM}]
    for t in conv:
        role = t.get("role", t.get("speaker", ""))
        txt = t.get("content", t.get("text", t.get("utterance", "")))
        if not txt:
            continue
        if role in ("player", "agent", "user", "human"):
            msgs.append({"role": "user", "content": str(txt)})
        else:
            msgs.append({"role": "assistant", "content": str(txt)})
    if len(msgs) < 3:
        return None
    eff = float(row.get("deal_efficiency", 0) or 0)
    rew = float(row.get("reward", 0) or 0)
    if eff < MIN_REWARD_KEEP and rew < -50:
        return None
    return {
        "messages": msgs,
        "deal_efficiency": eff,
        "reward": rew,
    }

out = [format_episode(dict(row)) for row in raw["train"]]
formatted = [x for x in out if x is not None]
print("episodes", len(formatted), "kept of", len(raw["train"]))

split = int(0.9 * len(formatted))
train_data = Dataset.from_list(formatted[:split])
eval_data  = Dataset.from_list(formatted[split:])


episodes 140 kept of 140


## Step 5 — Model + LoRA (Unsloth)


In [5]:
import os
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
    disable_log_stats=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Will map <|im_end|> to EOS = <|im_end|>.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Step 6 — Tokenize


In [6]:
def tok_one(row):
    t = tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": t}

tr = train_data.map(tok_one, remove_columns=train_data.column_names)
ev = eval_data.map(tok_one, remove_columns=eval_data.column_names)


Map:   0%|          | 0/126 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

## Step 7 — SFT with TRL


In [8]:
import torch
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="checkpoints/sft",
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tr,
    eval_dataset=ev,
    args=args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)
trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/126 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/14 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 126 | Num Epochs = 2 | Total steps = 32
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.168761,1.070173
2,0.995521,0.991213


Unsloth: Restored added_tokens_decoder metadata in checkpoints/sft/checkpoint-16/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in checkpoints/sft/checkpoint-32/tokenizer_config.json.


TrainOutput(global_step=32, training_loss=1.20980966091156, metrics={'train_runtime': 202.4359, 'train_samples_per_second': 1.245, 'train_steps_per_second': 0.158, 'total_flos': 1520738024509440.0, 'train_loss': 1.20980966091156, 'epoch': 2.0})

## Step 8 — Plots + push adapter


In [9]:
import os, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from huggingface_hub import HfApi

os.makedirs("results", exist_ok=True)
log = trainer.state.log_history
losses = [x["loss"] for x in log if "loss" in x]
steps  = [x["step"] for x in log if "loss" in x]
if steps:
    plt.figure(figsize=(8,3))
    plt.plot(steps, losses, color="#c9a84c")
    plt.title("SFT loss")
    plt.savefig("results/sft_loss_curve.png", dpi=120, bbox_inches="tight")
    plt.close()

ADAPTER = "checkpoints/sft_adapter"
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
api = HfApi()
api.create_repo(OUTPUT_REPO, exist_ok=True, token=HF_TOKEN, repo_type="model")
api.upload_folder(folder_path=ADAPTER, repo_id=OUTPUT_REPO, token=HF_TOKEN)
print("https://huggingface.co/" + OUTPUT_REPO)


Unsloth: Restored added_tokens_decoder metadata in checkpoints/sft_adapter/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ft_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

https://huggingface.co/sh4shv4t/parlay-negotiator
